# Scraping Data Review Goodreads

Notebook ini digunakan untuk melakukan proses pengumpulan data review buku dari website Goodreads menggunakan teknik web scraping berbasis Selenium. Hasil scraping kemudian disimpan sebagai dataset mentah untuk tahap preprocessing dan modeling selanjutnya.


**Output:**
review_id_mentah.csv

File ini merupakan hasil scraping langsung dari Goodreads yang berisi kumpulan review buku.


**Kolom yang dihasilkan:**

- **review_id**  
  Identitas unik setiap review yang diambil dari halaman Goodreads.

- **review_text**  
  Isi teks review asli yang diperoleh dari website (belum melalui proses pembersihan atau preprocessing).

- **spoiler_label**  
  Label yang menunjukkan apakah review mengandung spoiler (1) atau tidak (0), berdasarkan informasi dari dataset atau hasil labeling awal.


Tahapan proses scraping:

1. Setup Environment  
   Menyiapkan library dan konfigurasi yang dibutuhkan untuk proses scraping.

2. Daftar URL Buku  
   Mengumpulkan daftar link halaman buku Goodreads yang akan di-scrape.

3. Ambil Resource via Selenium  
   Mengakses halaman web secara otomatis menggunakan Selenium untuk mengambil konten review.

4. Helper Function  
   Membuat fungsi bantu untuk mengekstrak data review dari halaman web.

5. Pipeline Scraping  
   Menjalankan proses scraping secara terstruktur dari awal hingga akhir untuk semua URL.

6. Simpan ke CSV  
   Menyimpan hasil scraping ke dalam file review_id_mentah.csv.

7. Statistik Dataset  
   Melakukan analisis awal seperti jumlah data dan distribusi review yang berhasil dikumpulkan.

## 1. Setup Environtment

In [2]:
!pip install selenium webdriver-manager selenium-wire blinker==1.7.0 langdetect pandas tqdm beautifulsoup4 requests

## b. Import & Konfigurasi

In [3]:
import time
import random
import re
import json
import requests
import pandas as pd
from tqdm.notebook import tqdm
from langdetect import detect, LangDetectException
from bs4 import BeautifulSoup

from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from seleniumwire import webdriver as sw_driver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service

# buat session yang bypass proxy selenium-wire
api_session = requests.Session()
api_session.trust_env = False

# konfigurasi 
OUTPUT_CSV     = "review_id_mentah.csv"
LIMIT_PER_PAGE = 30       # jumlah review per request ke API
MAX_PAGES      = None     # None = ambil semua halaman, int = batas halaman per buku
REQUEST_PAUSE  = (1, 3)   # jeda acak antar request (detik)

# GraphQL endpoint & API key — ditemukan via intercept selenium-wire
GRAPHQL_URL = "https://kxbwmqov6jgg3daaamb744ycu4.appsync-api.us-east-1.amazonaws.com/graphql"
API_KEY     = "da2-xpgsdydkbregjhpr6ejzqdhuwy"

GRAPHQL_HEADERS = {
    "x-api-key"   : API_KEY,
    "content-type": "application/json",
    "accept"      : "application/graphql-response+json,application/json;q=0.9",
    "origin"      : "https://www.goodreads.com",
    "referer"     : "https://www.goodreads.com/",
    "user-agent"  : "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36",
}

GRAPHQL_QUERY = """
query getReviews($filters: BookReviewsFilterInput!, $pagination: PaginationInput) {
  getReviews(filters: $filters, pagination: $pagination) {
    totalCount
    edges {
      node {
        id
        text
        spoilerStatus
        rating
        createdAt
        updatedAt
        creator {
          name
          webUrl
        }
      }
    }
    pageInfo {
      nextPageToken
    }
  }
}
"""

## 2. Daftar URL Buku 
Daftar ini berisi kumpulan link halaman buku dari Goodreads yang akan digunakan sebagai sumber data utama untuk proses scraping review. Setiap URL merepresentasikan satu buku yang akan diambil review-nya secara otomatis menggunakan Selenium.

Sebelum digunakan, dilakukan proses filtering untuk memastikan semua link valid (dimulai dengan http), sehingga hanya URL yang benar-benar aktif yang diproses dalam pipeline scraping.

Jumlah total buku yang akan di-scrape kemudian ditampilkan untuk memastikan seluruh dataset sumber sudah terdefinisi dengan benar sebelum proses pengambilan data dimulai.

In [4]:
goodreads_links = [
    "https://www.goodreads.com/book/show/133227651-angsa-dan-kelelawar",                    
    "https://www.goodreads.com/book/show/25924054-the-dead-returns",                       
    "https://www.goodreads.com/book/show/123850391-teka-teki-rumah-aneh",                  
    "https://www.goodreads.com/en/book/show/205345677-pasien",                              
    "https://www.goodreads.com/book/show/20613611-malice",                                  
    "https://www.goodreads.com/book/show/1321926.The_Tokyo_Zodiac_Murders",                
    "https://www.goodreads.com/book/show/52699713-second-sister",                           
    "https://www.goodreads.com/en/book/show/36804340-the-good-son",                         
    "https://www.goodreads.com/en/book/show/22432940-girls-in-the-dark",                   
    "https://www.goodreads.com/book/show/32450412-holy-mother",                             
    "https://www.goodreads.com/book/show/204903103-tragedi-pedang-keadilan",                
    "https://www.goodreads.com/en/book/show/23847971-a-midsummer-s-equation",               
    "https://www.goodreads.com/book/show/19161835-confessions",                             
    "https://www.goodreads.com/book/show/2016000.Rahasia_Meede",                           
    "https://goodreads.com/book/show/50078136-dua-dini-hari",                             
    "https://www.goodreads.com/book/show/25082874-misteri-patung-garam",                   
    "https://www.goodreads.com/book/show/34300076-24-jam-bersama-gaspar",                   
    "https://www.goodreads.com/book/show/15990354-omen",                                    
    "https://www.goodreads.com/book/show/18000845-tujuh-lukisan-horor",
    "https://www.goodreads.com/book/show/18493770-misteri-organisasi-rahasia-the-judges",
    "https://www.goodreads.com/book/show/20581395-malam-karnaval-berdarah",
    "https://www.goodreads.com/book/show/22325046-kutukan-hantu-opera",
    "https://www.goodreads.com/book/show/23261061-sang-pengkhianat",
    "https://www.goodreads.com/book/show/25967996-target-terakhir",
    "https://www.goodreads.com/book/show/9475091-obsesi",
    "https://www.goodreads.com/book/show/11355995-pengurus-mos-harus-mati",
    "https://www.goodreads.com/book/show/12955672-permainan-maut",
    "https://www.goodreads.com/book/show/13637705-teror",
]

goodreads_links = [url for url in goodreads_links if url.startswith("http")]
print(f"Total buku: {len(goodreads_links)}")

Total buku: 28


## 3. Ambil resourceId via Selenium

Selenium dipakai hanya sekali per buku untuk mengambil resourceId (format kca://work/...) yang dibutuhkan sebagai filter GraphQL.
Cara: intercept request GraphQL pertama yang otomatis dipanggil saat halaman review dibuka.

In [5]:
def build_sw_driver():
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1440,900")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36"
    )
    options.add_experimental_option("excludeSwitches", ["enable-automation"])  
    options.add_experimental_option("useAutomationExtension", False)          
    service = Service(ChromeDriverManager().install())
    return sw_driver.Chrome(service=service, options=options)

## 4. Helper Functions
Helper functions ini berisi kumpulan fungsi pendukung yang digunakan selama proses scraping data review dari Goodreads. Fungsi-fungsi tersebut mencakup pengecekan bahasa untuk memastikan hanya review berbahasa Indonesia yang digunakan, pembersihan teks dari HTML dan karakter tidak relevan, pengambilan data review dari GraphQL API dengan sistem pagination dan retry untuk menjaga kestabilan request, serta ekstraksi resourceId menggunakan Selenium melalui intercept request halaman review. Seluruh fungsi ini bekerja sebagai pipeline pendukung utama untuk memastikan data yang diambil bersih, valid, dan dapat digunakan untuk tahap analisis berikutnya.

In [6]:
# Filter bahasa Indonesia 
def is_indonesian(text, min_chars=30):
    """Return True jika teks terdeteksi sebagai bahasa Indonesia."""
    if not text or len(text.strip()) < min_chars:
        return False
    try:
        return detect(text) == "id"
    except LangDetectException:
        return False

#  Bersihkan teks HTML dari API 
def clean_text(html_text):
    if not html_text or len(html_text.strip()) < 3:
        return ""
    text = re.sub(r"</?spoiler>", "", html_text, flags=re.I)
    # Hanya parse BeautifulSoup kalau mengandung HTML tag
    if "<" in text:
        text = BeautifulSoup(text, "html.parser").get_text(separator=" ", strip=True)
    return re.sub(r"\s+", " ", text).strip()

#  Fetch satu halaman review dari GraphQL API 
def fetch_reviews_page(resource_id, cursor=None, limit=30):
    pagination = {"limit": limit}
    if cursor:
        pagination["after"] = cursor

    payload = {
        "operationName": "getReviews",
        "variables": {
            "filters": {
                "resourceType": "WORK",
                "resourceId": resource_id
            },
            "pagination": pagination
        },
        "extensions": {"clientLibrary": {"name": "@apollo/client", "version": "4.1.6"}},
        "query": GRAPHQL_QUERY
    }

    for attempt in range(3):  # retry 3x
        try:
            resp = api_session.post(GRAPHQL_URL, headers=GRAPHQL_HEADERS, json=payload, timeout=30)
            data = resp.json()
            result = data.get("data", {}).get("getReviews")
            if result is None:
                print(f"  [API ERR] {data.get('errors', 'unknown')}")
                return None, None, None
            return result["edges"], result["pageInfo"]["nextPageToken"], result["totalCount"]
        except Exception as e:
            print(f"  [REQ ERR attempt {attempt+1}] {e}")
            time.sleep(5)

    return None, None, None

def get_resource_id(driver, book_url):
    """
    Buka halaman buku → navigasi ke halaman review → intercept request GraphQL
    → ekstrak resourceId dari body request.
    Return: resourceId string atau None jika gagal.
    """
    book_name = book_url.split("/")[-1].replace("-", " ").title()

    driver.requests.clear()
    driver.get(book_url)
    time.sleep(3)

    try:
        btn = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((
                By.XPATH,
                "//a[contains(@href, '/reviews')]"
            ))
        )
        reviews_url = btn.get_attribute("href")

        if "goodreads.com" not in reviews_url:
            print(f"[{book_name}] URL bukan Goodreads reviews: {reviews_url}")
            return None

    except TimeoutException:
        print(f"[{book_name}] Tombol reviews tidak ditemukan")
        return None

    driver.requests.clear()
    driver.get(reviews_url)
    time.sleep(5)

    for req in driver.requests:
        if "appsync" in req.url and req.body:
            try:
                body = json.loads(req.body.decode("utf-8"))
                if body.get("operationName") == "getReviews":
                    resource_id = body["variables"]["filters"]["resourceId"]
                    print(f"[{book_name}] resourceId: {resource_id}")
                    return resource_id
            except Exception:
                continue

    print(f"[{book_name}] Tidak bisa ekstrak resourceId")
    return None

## 5. Pipeline Utama
Pipeline ini digunakan untuk mengumpulkan data review dari Goodreads secara bertahap dan terstruktur. Proses dimulai dengan mengambil resourceId dari setiap buku menggunakan Selenium dan menyimpannya ke file JSON agar dapat digunakan ulang tanpa perlu scraping dari awal. Setelah itu, data review diambil langsung melalui GraphQL API berdasarkan resourceId, kemudian setiap review dibersihkan, difilter hanya untuk bahasa Indonesia, diberi label spoiler, dan dihindari dari duplikasi. Seluruh hasil scraping disimpan secara berkala ke file CSV sebagai checkpoint hingga semua buku selesai diproses dan dataset akhir berhasil terbentuk.

In [ ]:
# ambil semua resourceId dulu, simpan ke JSON 
import json
import os

RESOURCE_MAP_FILE = "resource_map.json"

# kalau file sudah ada, load langsung, tidak perlu ulang
if os.path.exists(RESOURCE_MAP_FILE):
    with open(RESOURCE_MAP_FILE) as f:
        resource_map = json.load(f)
    print(f"Resource map loaded: {len(resource_map)} buku")
else:
    resource_map = {}
    for book_url in tqdm(goodreads_links, desc="Ambil resourceId"):
        if book_url in resource_map:
            continue
        book_name = book_url.split("/")[-1].replace("-", " ").title()
        driver = build_sw_driver()
        resource_id = get_resource_id(driver, book_url)
        driver.quit()
        if resource_id:
            resource_map[book_url] = resource_id
            print(f"OK: {book_name}")
        else:
            print(f"GAGAL: {book_url}")
        time.sleep(2)

    with open(RESOURCE_MAP_FILE, "w") as f:
        json.dump(resource_map, f, indent=2)
    print(f"Tersimpan: {len(resource_map)} resourceId")

# scraping GraphQL murni tanpa Selenium 
all_reviews = []
seen_ids = set()

for book_url, resource_id in tqdm(resource_map.items(), desc="Scraping"):
    print(f"\nMemproses: {book_url}")

    cursor = None
    page = 0
    book_count = 0

    while True:
        if MAX_PAGES is not None and page >= MAX_PAGES:
            print(f"Batas halaman ({MAX_PAGES}) tercapai.")
            break

        edges, next_cursor, total = fetch_reviews_page(resource_id, cursor, LIMIT_PER_PAGE)
        if edges is None:
            print("Gagal fetch, hentikan buku ini.")
            break

        if page == 0:
            print(f"Total review di Goodreads: {total}")

        new_this_page = 0
        for edge in edges:
            node = edge["node"]
            if node is None:
                continue
            review_id = node.get("id", "")
            if review_id in seen_ids:
                continue
            clean = clean_text(node.get("text") or "")
            if not is_indonesian(clean):
                continue
            spoiler_label = 1 if node.get("spoilerStatus") else 0
            seen_ids.add(review_id)
            all_reviews.append({
                "book_url"     : book_url,
                "review_id"    : review_id,
                "reviewer"     : node["creator"]["name"] if node.get("creator") else "",
                "reviewer_url" : node["creator"]["webUrl"] if node.get("creator") else "",
                "rating"       : node.get("rating"),
                "created_at"   : node.get("createdAt"),
                "review_text"  : clean,
                "spoiler_label": spoiler_label,
            })
            new_this_page += 1
            book_count += 1

        page += 1
        print(f"Halaman {page}: {len(edges)} dari API, {new_this_page} review bahasa Indonesia baru (total review bahasa Indonesia: {book_count})")

        if not next_cursor:
            print("Selesai.")
            break

        cursor = next_cursor
        time.sleep(random.uniform(*REQUEST_PAUSE))

    # simpan setiap selesai 1 buku
    df_temp = pd.DataFrame(all_reviews)
    df_temp.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"Checkpoint disimpan: {len(df_temp)} review")

print(f"\nTotal review Indonesia terkumpul: {len(all_reviews)}")

Resource map loaded: 28 buku


Scraping:   0%|          | 0/28 [00:00<?, ?it/s]


Memproses: https://www.goodreads.com/book/show/133227651-angsa-dan-kelelawar
Total review di Goodreads: 650
Halaman 1: 30 dari API, 6 review bahasa Indonesia baru (total review bahasa Indonesia: 6)
Halaman 2: 30 dari API, 6 review bahasa Indonesia baru (total review bahasa Indonesia: 12)
Halaman 3: 30 dari API, 16 review bahasa Indonesia baru (total review bahasa Indonesia: 28)
Halaman 4: 30 dari API, 10 review bahasa Indonesia baru (total review bahasa Indonesia: 38)
Halaman 5: 30 dari API, 10 review bahasa Indonesia baru (total review bahasa Indonesia: 48)
Halaman 6: 30 dari API, 21 review bahasa Indonesia baru (total review bahasa Indonesia: 69)
Halaman 7: 30 dari API, 17 review bahasa Indonesia baru (total review bahasa Indonesia: 86)
Halaman 8: 30 dari API, 19 review bahasa Indonesia baru (total review bahasa Indonesia: 105)
Halaman 9: 30 dari API, 13 review bahasa Indonesia baru (total review bahasa Indonesia: 118)
Halaman 10: 30 dari API, 20 review bahasa Indonesia baru (total 

## 6. Simpan ke CSV

In [ ]:
df = pd.DataFrame(all_reviews)
if not df.empty:
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"Disimpan: {len(df)} review")
    print(f"Spoiler: {df['spoiler_label'].sum()}")
    print(f"Non-spoiler: {(df['spoiler_label']==0).sum()}")
    print(f"Dari berapa buku: {df['book_url'].nunique()}")
else:
    print("Tidak ada data.")

## 7. Statistik Dataset
Proses ini dilakukan untuk melihat gambaran awal dari dataset hasil scraping sebelum masuk ke tahap preprocessing dan modeling. Analisis mencakup distribusi label spoiler dan non-spoiler untuk melihat keseimbangan data, distribusi jumlah review pada setiap buku untuk mengetahui sebaran data per sumber, serta analisis panjang teks review untuk memahami karakteristik umum data seperti rata-rata panjang dan variasinya. Hasil statistik ini membantu memastikan kualitas dataset serta memberikan insight awal terhadap kondisi data yang akan digunakan pada tahap selanjutnya

In [ ]:
if not df.empty:
    print("Distribusi Label")
    print(df["spoiler_label"].value_counts().rename({0: "Non-Spoiler", 1: "Spoiler"}))
    print()
    print("Distribusi per Buku")
    print(
        df.groupby("book_url")["spoiler_label"]
          .agg(total="count", spoiler="sum")
          .assign(pct_spoiler=lambda x: (x["spoiler"] / x["total"] * 100).round(1))
    )
    print()
    print("Panjang Teks Review")
    df["text_len"] = df["review_text"].str.len()
    print(df["text_len"].describe().round(1))